# BroadbandCare GRPO-Only Training Notebook (Colab)

This notebook performs **GRPO refinement only** using environment-aligned rewards:
1. Build GRPO prompt dataset from BroadbandCare cases
2. Load `Qwen2.5-1.5B-Instruct` with Unsloth (4-bit + LoRA)
3. Train with `TRL` GRPO trainer + custom reward function
4. Evaluate policy by running episodes in `BroadbandSupportEnv`

> No SFT stage is used in this notebook.

In [ ]:
!pip install -q -U pip
!pip install -q unsloth trl transformers accelerate bitsandbytes datasets matplotlib pandas

In [ ]:
import os
import sys
import json
import re
import random
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset

# If running in Colab after cloning, cd into repo root before running cells.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "env").exists():
    raise RuntimeError("Run this notebook from repository root (folder containing env/, data/, models.py).")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from env.env import BroadbandSupportEnv
from models import BroadbandcareAction

print("Project root:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Build GRPO prompt dataset (no supervised targets)
cases_path = PROJECT_ROOT / "data" / "cases.json"
with cases_path.open("r", encoding="utf-8") as f:
    cases = json.load(f)

AVAILABLE_TOOLS = [
    "get_account_details",
    "get_user_broadband_location",
    "run_speed_test",
    "get_ping_stats",
    "get_router_stats",
    "search_troubleshooting_docs",
    "change_dns_settings",
    "restart_router",
    "create_support_ticket",
    "resolve_ticket",
]

CASE_BY_ID = {c["case_id"]: c for c in cases}


def build_prompt(case_id: str, customer_message: str, history: list, last_result: dict) -> str:
    return (
        f"Case ID: {case_id}\n"
        f"Customer Issue: {customer_message}\n\n"
        f"Steps Taken So Far:\n{json.dumps(history, ensure_ascii=True)}\n\n"
        f"Last Tool Result:\n{json.dumps(last_result, ensure_ascii=True)}\n\n"
        f"Available Tools: {AVAILABLE_TOOLS}\n\n"
        "Respond ONLY with valid JSON:\n"
        '{"tool": "<tool_name>", "args": {}}'
    )


# Build prompts at multiple history depths so GRPO learns sequencing, not only first-step action.
prompts = []
for case in cases:
    solution = case["solution_path"]
    for depth in range(len(solution)):
        history_tools = solution[:depth]
        history = [{"tool": t, "args": {}} for t in history_tools]
        last_result = {"ok": True, "message": "Episode started."} if depth == 0 else {"ok": True, "tool": history_tools[-1]}

        prompts.append({
            "prompt": build_prompt(
                case_id=case["case_id"],
                customer_message=case["customer_message"],
                history=history,
                last_result=last_result,
            )
        })

grpods = Dataset.from_pandas(pd.DataFrame(prompts))
print("GRPO prompts:", len(grpods))
print("Example prompt snippet:\n", grpods[0]["prompt"][:300], "...")

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

# Critical for GRPO with Unsloth: patches trainer/loss plumbing.
PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOConfig, GRPOTrainer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

FastLanguageModel.for_training(model)
print("Model + LoRA ready for GRPO")

In [ ]:
json_re = re.compile(r"\{[\s\S]*\}")


def extract_case_id(prompt: str) -> str | None:
    m = re.search(r"Case ID:\s*([A-Z0-9\-]+)", prompt)
    return m.group(1) if m else None


def extract_history_tools(prompt: str) -> list:
    m = re.search(r"Steps Taken So Far:\n(.*?)\n\nLast Tool Result:", prompt, re.DOTALL)
    if not m:
        return []
    raw = m.group(1).strip()
    if not raw:
        return []
    try:
        arr = json.loads(raw)
    except Exception:
        return []
    tools = []
    if isinstance(arr, list):
        for x in arr:
            if isinstance(x, dict) and isinstance(x.get("tool"), str):
                tools.append(x["tool"])
    return tools


def parse_tool(completion: str) -> str | None:
    m = json_re.search(completion)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except Exception:
        return None
    return obj.get("tool") if isinstance(obj, dict) else None


def next_expected_tool(solution_path: list, history_tools: list) -> str | None:
    idx = 0
    for t in history_tools:
        if idx < len(solution_path) and t == solution_path[idx]:
            idx += 1
    if idx >= len(solution_path):
        return None
    return solution_path[idx]


def grpo_reward_func(prompts, completions, **kwargs):
    """
    Sequence-aware GRPO reward:
      +1.5 correct next step
      -1.0 wrong step
      -1.5 invalid/missing JSON tool
      -2.0 invalid tool name (not allowed)
      +0.4 bonus for choosing resolve_ticket only at terminal expected step
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        case_id = extract_case_id(prompt)
        tool = parse_tool(completion)
        history_tools = extract_history_tools(prompt)

        if (case_id is None) or (case_id not in CASE_BY_ID):
            rewards.append(-1.5)
            continue
        if tool is None:
            rewards.append(-1.5)
            continue
        if tool not in AVAILABLE_TOOLS:
            rewards.append(-2.0)
            continue

        case = CASE_BY_ID[case_id]
        expected = next_expected_tool(case["solution_path"], history_tools)

        if expected is None:
            rewards.append(0.4 if tool == "resolve_ticket" else -1.0)
            continue

        if tool == expected:
            rewards.append(1.5)
        else:
            rewards.append(-1.0)

    return rewards


grpo_config = GRPOConfig(
    output_dir="./grpo_broadbandcare",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=8,
    logging_steps=5,
    max_prompt_length=768,
    max_completion_length=96,
    beta=0.02,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=grpo_reward_func,
    train_dataset=grpods,
    args=grpo_config,
)

print("GRPO trainer configured (sequence-aware reward)")

In [ ]:
train_result = trainer.train()

rows = []
for row in trainer.state.log_history:
    rows.append({
        "step": row.get("step", None),
        "epoch": row.get("epoch", None),
        "loss": row.get("loss", None),
        "reward": row.get("reward", row.get("rewards", None)),
        "kl": row.get("kl", None),
    })

train_logs_df = pd.DataFrame(rows)
print(train_result)
train_logs_df.tail()

In [ ]:
if len(train_logs_df) == 0:
    print("No GRPO logs captured.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    if train_logs_df["loss"].notna().any():
        d = train_logs_df.dropna(subset=["loss"])
        ax[0].plot(d["step"], d["loss"], marker="o")
        ax[0].set_title("GRPO Loss")
        ax[0].set_xlabel("Step")
        ax[0].set_ylabel("Loss")
    else:
        ax[0].text(0.1, 0.5, "No loss logged")
        ax[0].set_title("GRPO Loss")

    if train_logs_df["reward"].notna().any():
        d = train_logs_df.dropna(subset=["reward"])
        ax[1].plot(d["step"], d["reward"], marker="o")
        ax[1].set_title("GRPO Reward")
        ax[1].set_xlabel("Step")
        ax[1].set_ylabel("Reward")
    else:
        ax[1].text(0.1, 0.5, "No reward logged")
        ax[1].set_title("GRPO Reward")

    for a in ax:
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
FastLanguageModel.for_inference(model)

json_re = re.compile(r"\{[\s\S]*\}")

def choose_action_from_model(obs):
    # Case ID is unknown at runtime; pass a neutral marker.
    prompt = build_prompt(
        case_id="EVAL",
        customer_message=obs.customer_message,
        history=obs.history,
        last_result=obs.last_result,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            temperature=0.0,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    m = json_re.search(text)
    if not m:
        return {"tool": "get_account_details", "args": {}}
    try:
        parsed = json.loads(m.group(0))
    except Exception:
        return {"tool": "get_account_details", "args": {}}

    tool = parsed.get("tool", "get_account_details")
    args = parsed.get("args", {}) if isinstance(parsed.get("args", {}), dict) else {}
    if tool not in AVAILABLE_TOOLS:
        tool = "get_account_details"
    return {"tool": tool, "args": args}


def rollout_eval(n_episodes=20, seed=123):
    rng = random.Random(seed)
    rows = []
    for _ in range(n_episodes):
        env = BroadbandSupportEnv(seed=rng.randint(1, 10_000_000))
        obs = env.reset()
        total_reward = 0.0
        while not obs.done and obs.step_count < obs.max_steps:
            action = choose_action_from_model(obs)
            obs = env.step(BroadbandcareAction(tool=action["tool"], args=action["args"]))
            total_reward += obs.reward
        rows.append({
            "episode_reward": total_reward,
            "steps": obs.step_count,
            "tool_accuracy": obs.metadata.get("tool_accuracy", 0.0),
            "path_coverage": obs.metadata.get("path_coverage", 0.0),
            "resolved": obs.metadata.get("resolved", 0.0),
        })
    return pd.DataFrame(rows)


results_df = rollout_eval(n_episodes=20)
print(results_df.describe())
print("\nMean episode reward:", results_df["episode_reward"].mean())
print("Mean tool accuracy:", results_df["tool_accuracy"].mean())
print("Mean resolution rate:", results_df["resolved"].mean())